# Testing LangGraph Agents with fasteval

This notebook demonstrates end-to-end evaluation of LangGraph agents using the `fasteval-langgraph` harness.

**What you'll learn:**

1. **Setup** — Install and configure `fasteval-langgraph`
2. **Sample Agent** — Build a multi-node support agent
3. **First Evaluation** — Wrap with `harness()`, use `.chat()`, score with `fe.score()`
4. **Route Verification** — Assert correct node trajectories
5. **Tool Trajectory** — Evaluate tool call accuracy and sequence
6. **Combined Metrics** — Stack tool + output quality metrics

Every cell is **runnable**. Cells that use LLM-as-judge metrics require an OpenAI API key.

---

## Setup

In [ ]:
# Install fasteval with LangGraph plugin
!pip install -q fasteval-langgraph

In [ ]:
import os

# --- Set your OpenAI API key (required for LLM-as-judge metrics) ---
#
# Option 1: Set it directly (uncomment and paste your key)
# os.environ["OPENAI_API_KEY"] = "sk-..."
#
# Option 2: Colab Secrets (recommended)
#   1. Click the key icon in the left sidebar
#   2. Add a secret named OPENAI_API_KEY with your key
#   3. Toggle "Notebook access" on
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab Secrets.")
except (ImportError, Exception):
    if os.environ.get("OPENAI_API_KEY"):
        print("API key found in environment.")
    else:
        print(
            "WARNING: OPENAI_API_KEY not set.\n"
            "LLM-as-judge metrics (correctness, relevance) will fail.\n"
            "Route verification and deterministic checks still work."
        )

In [ ]:
import fasteval as fe
from fasteval_langgraph import harness
from fasteval.models import ExpectedTool, ToolCall

print("fasteval + langgraph harness ready.")

---

## 1. Sample LangGraph Agent

We'll build a multi-node support agent that classifies user intent and routes to different backends:

- **classifier** — Determines if the query is FAQ or troubleshooting
- **rag** — Retrieves documents for FAQ queries
- **planner** — Creates step-by-step plans for troubleshooting
- **responder** — Generates the final response

> **Collab Note:** This agent uses deterministic routing for reproducibility. In production, the classifier would be an LLM call.

In [ ]:
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from langchain_core.messages import AIMessage, HumanMessage


class SupportState(MessagesState):
    intent: str
    docs: list
    plan: str


def classifier(state: SupportState) -> Command:
    last_msg = state["messages"][-1].content
    if "troubleshoot" in last_msg.lower():
        intent = "TROUBLESHOOTING"
    else:
        intent = "FAQ"
    return Command(
        update={"intent": intent},
        goto="rag" if intent == "FAQ" else "planner",
    )


def rag(state: SupportState) -> Command:
    return Command(
        update={"docs": [f"Retrieved docs for: {state['intent']}"]},
        goto="responder",
    )


def planner(state: SupportState) -> Command:
    return Command(
        update={"plan": f"Step-by-step plan for: {state['intent']}"},
        goto="responder",
    )


def responder(state: SupportState) -> dict:
    context = state.get("docs") or [state.get("plan", "")]
    return {
        "messages": [AIMessage(content=f"Here's your answer based on: {context[0]}")],
    }


def build_support_agent():
    builder = StateGraph(SupportState)
    builder.add_node("classifier", classifier)
    builder.add_node("rag", rag)
    builder.add_node("planner", planner)
    builder.add_node("responder", responder)
    builder.add_edge(START, "classifier")
    builder.add_edge("responder", END)
    return builder.compile(checkpointer=MemorySaver())


agent = build_support_agent()
print(f"Agent built with nodes: {[n for n in agent.nodes if not n.startswith('__')]}")

---

## 2. Your First Evaluation

Wrap the compiled graph with `harness()` and use `.chat()` to send messages.

The harness:
- Auto-detects `MessagesState` and configures input/output handling
- Creates a new thread for each `.chat()` call
- Captures a full execution trace via streaming
- Returns a `ChatResult` with response, filtered state, and trace

> **Collab Note:** `harness()` is the preferred entry point. It's a thin wrapper around `GraphHarness()` with sensible defaults.

In [ ]:
graph = harness(agent)

# Inspect what the harness detected
print(f"Has MessagesState: {graph.has_messages_state}")
print(f"Available nodes:   {graph.nodes}")

In [ ]:
import asyncio


async def run_first_eval():
    result = await graph.chat("What is OAuth?")

    print(f"Response:   {result.response}")
    print(f"Nodes ran:  {result.nodes_ran}")
    print(f"State:      {result.state}")
    print(f"Trace steps: {len(result.trace)}")
    for step in result.trace:
        print(f"  -> {step.node}: {dict(step.updates)}")

    return result


result = await run_first_eval()

---

## 3. Route Verification

Use `nodes_ran` and `state` to verify your agent takes the correct path.

This is a deterministic check -- no LLM judge needed.

> **Collab Note:** Route verification is the fastest feedback loop for agent development. Start here before adding LLM-as-judge metrics.

In [ ]:
async def verify_routes():
    # FAQ route: classifier -> rag -> responder
    faq_result = await graph.chat("What is OAuth?")
    assert "classifier" in faq_result.nodes_ran
    assert "rag" in faq_result.nodes_ran
    assert "responder" in faq_result.nodes_ran
    assert "planner" not in faq_result.nodes_ran
    assert faq_result.state["intent"] == "FAQ"
    print("FAQ route:            classifier -> rag -> responder  PASSED")

    # Troubleshooting route: classifier -> planner -> responder
    ts_result = await graph.chat("I need to troubleshoot my login")
    assert "classifier" in ts_result.nodes_ran
    assert "planner" in ts_result.nodes_ran
    assert "responder" in ts_result.nodes_ran
    assert "rag" not in ts_result.nodes_ran
    assert ts_result.state["intent"] == "TROUBLESHOOTING"
    print("Troubleshoot route:   classifier -> planner -> responder  PASSED")


await verify_routes()

---

## 4. Tool Trajectory Testing

When your agent uses LangChain tools, extract tool calls from the trace and evaluate with fasteval's tool metrics:

- `@fe.tool_call_accuracy` — Were the right tools called?
- `@fe.tool_sequence` — Were they called in the right order?
- `@fe.tool_args_match` — Were the right arguments passed?

> **Collab Note:** Our sample agent doesn't use LangChain tools, so we demonstrate the pattern with simulated tool calls. In a real agent, you'd extract them from `result.trace`.

In [ ]:
# Simulated tool calls (in a real agent, extract from result.trace)
simulated_tool_calls = [
    ToolCall(name="get_weather", arguments={"city": "London"}),
]

expected_tools = [
    ExpectedTool(name="get_weather", args={"city": "London"}),
]


@fe.tool_call_accuracy(threshold=0.9)
def test_tool_call_accuracy():
    """Verify the agent calls the right tools."""
    return fe.score(
        actual_output="The weather in London is 65F and cloudy.",
        tool_calls=simulated_tool_calls,
        expected_tools=expected_tools,
        input="What's the weather in London?",
    )


result = test_tool_call_accuracy()
print(f"Passed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

In [ ]:
# Simulated multi-step tool calls
multi_step_calls = [
    ToolCall(name="search_web", arguments={"query": "Tokyo population"}),
    ToolCall(name="calculate", arguments={"expression": "13960000 * 0.1"}),
]

expected_sequence = [
    ExpectedTool(name="search_web"),
    ExpectedTool(name="calculate"),
]


@fe.tool_sequence(threshold=0.9)
def test_tool_sequence():
    """Verify tools are called in the correct order."""
    return fe.score(
        actual_output="10% of Tokyo's population is approximately 1,396,000.",
        tool_calls=multi_step_calls,
        expected_tools=expected_sequence,
        input="Search for Tokyo's population, then calculate 10% of it",
    )


result = test_tool_sequence()
print(f"Passed: {result.passed}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    print(f"  {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## 5. Combining Tool and Output Metrics

Stack multiple decorators to evaluate both tool usage AND response quality in a single test.

> **Collab Note:** This is the most realistic evaluation pattern. In production, you want to verify that the agent not only calls the right tools but also produces a high-quality response from the tool results.

In [ ]:
@fe.tool_call_accuracy(threshold=0.9)
@fe.tool_args_match(threshold=0.8)
@fe.correctness(threshold=0.7)
def test_combined_evaluation():
    """Evaluate tool trajectory and output quality together."""
    tool_calls = [
        ToolCall(name="calculate", arguments={"expression": "100 + 200 + 300"}),
    ]
    expected = [
        ExpectedTool(name="calculate", args={"expression": "100 + 200 + 300"}),
    ]

    return fe.score(
        actual_output="The result of 100 + 200 + 300 is 600.",
        expected_output="The result is 600",
        tool_calls=tool_calls,
        expected_tools=expected,
        input="What is 100 + 200 + 300?",
    )


result = test_combined_evaluation()
print(f"\nOverall: {'PASSED' if result.passed else 'FAILED'}  |  Score: {result.aggregate_score:.2f}")
for m in result.metric_results:
    status = 'PASS' if m.passed else 'FAIL'
    print(f"  [{status}] {m.metric_name}: {m.score:.2f} (threshold {m.threshold})")

---

## Next Steps

This notebook covered the end-to-end workflow. Dive deeper with these companion notebooks:

| Notebook | What you'll learn |
|----------|------------------|
| [Harness Basics](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/harness-basics.ipynb) | `.chat()`, `ChatResult`, hooks, auto-detection, graph introspection |
| [Multi-turn Sessions](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/sessions.ipynb) | `.session()`, conversation metrics, interrupt/resume |
| [Node Testing & Mocking](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langgraph-testing/node-mocking.ipynb) | `.node().run()`, `mock()`, isolated testing patterns |

For the full API reference, see the [fasteval-langgraph plugin docs](https://fasteval.dev/docs/plugins/langgraph/overview).